In [16]:
!unzip ExDark.zip

Streaming output truncated to the last 5000 lines.
  inflating: ExDark/Motorbike/2017_07361.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2017_07361.jpg  
  inflating: ExDark/Motorbike/2015_06160.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_06160.jpg  
  inflating: ExDark/Motorbike/2015_05864.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_05864.jpg  
  inflating: ExDark/Motorbike/2015_05870.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_05870.jpg  
  inflating: ExDark/Motorbike/2015_05858.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_05858.jpg  
  inflating: ExDark/Motorbike/2015_05874.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_05874.jpg  
  inflating: ExDark/Motorbike/2015_05860.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_05860.jpg  
  inflating: ExDark/Motorbike/2015_05848.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_05848.jpg  
  inflating: ExDark/Motorbike/2015_06164.jpg  
  inflating: __MACOSX/ExDark/Motorbike/._2015_06164.jpg  
  in

In [32]:
CLASSES = [
    "Bicycle","Boat","Bottle","Bus","Car",
    "Cat","Chair","Cup","Dog","Motorbike",
    "People","Table"
]

CLS2IDX = {c:i for i,c in enumerate(CLASSES)}

print(CLS2IDX)

{'Bicycle': 0, 'Boat': 1, 'Bottle': 2, 'Bus': 3, 'Car': 4, 'Cat': 5, 'Chair': 6, 'Cup': 7, 'Dog': 8, 'Motorbike': 9, 'People': 10, 'Table': 11}


In [42]:
import cv2
import shutil
from pathlib import Path

CLASSES = [
    "Bicycle","Boat","Bottle","Bus","Car",
    "Cat","Chair","Cup","Dog","Motorbike",
    "People","Table"
]

CLS2IDX = {c:i for i,c in enumerate(CLASSES)}

def convert_exdark_final(img_root, ann_root, output):

    img_root = Path(img_root)
    ann_root = Path(ann_root)
    output = Path(output)

    (output/"images/train").mkdir(parents=True, exist_ok=True)
    (output/"labels/train").mkdir(parents=True, exist_ok=True)

    count = 0

    for cls_folder in img_root.iterdir():

        if not cls_folder.is_dir():
            continue

        class_name = cls_folder.name

        for img_path in cls_folder.glob("*"):

            if not img_path.is_file():
                continue

            # 🔥 CORRECT PATH (THIS WAS YOUR BUG)
            ann_path = ann_root / class_name / (img_path.name + ".txt")

            if not ann_path.exists():
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            h, w = img.shape[:2]
            boxes = []

            with open(ann_path) as f:
                for line in f:

                    line = line.strip()

                    if line.startswith("%") or line == "":
                        continue

                    parts = line.split()

                    if len(parts) < 5:
                        continue

                    cls = parts[0]

                    if cls not in CLS2IDX:
                        continue

                    cls_id = CLS2IDX[cls]

                    x, y, bw, bh = map(float, parts[1:5])

                    cx = (x + bw/2) / w
                    cy = (y + bh/2) / h
                    bw /= w
                    bh /= h

                    boxes.append((cls_id, cx, cy, bw, bh))

            if len(boxes) == 0:
                continue

            shutil.copy(img_path, output/"images/train"/img_path.name)

            label_file = output/"labels/train"/(img_path.stem + ".txt")

            with open(label_file, "w") as f:
                for b in boxes:
                    f.write(f"{b[0]} {b[1]} {b[2]} {b[3]} {b[4]}\n")

            count += 1

    print("✅ Valid labeled images:", count)

In [43]:
convert_exdark_final(
    "ExDark",
    "ExDark_Annno",
    "ExDark_YOLO"
)

✅ Valid labeled images: 7361


In [44]:
!zip -r ExDark_YOLO.zip ExDark_YOLO

Streaming output truncated to the last 5000 lines.
  adding: ExDark_YOLO/labels/train/2015_05471.txt (deflated 57%)
  adding: ExDark_YOLO/labels/train/2015_01086.txt (deflated 14%)
  adding: ExDark_YOLO/labels/train/2015_02800.txt (deflated 62%)
  adding: ExDark_YOLO/labels/train/2015_06499.txt (deflated 15%)
  adding: ExDark_YOLO/labels/train/2015_05016.txt (deflated 39%)
  adding: ExDark_YOLO/labels/train/2015_01875.txt (deflated 66%)
  adding: ExDark_YOLO/labels/train/2015_04543.txt (deflated 48%)
  adding: ExDark_YOLO/labels/train/2015_02641.txt (deflated 49%)
  adding: ExDark_YOLO/labels/train/2015_02516.txt (deflated 57%)
  adding: ExDark_YOLO/labels/train/2015_04291.txt (deflated 47%)
  adding: ExDark_YOLO/labels/train/2015_02165.txt (deflated 38%)
  adding: ExDark_YOLO/labels/train/2015_05991.txt (deflated 41%)
  adding: ExDark_YOLO/labels/train/2015_01857.txt (deflated 68%)
  adding: ExDark_YOLO/labels/train/2015_04259.txt (deflated 58%)
  adding: ExDark_YOLO/labels/train/2015

In [45]:
from google.colab import files
files.download("ExDark_YOLO.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>